In [1]:
from pathlib import Path
from pptx import Presentation
from docx import Document
from unstructured.partition.auto import partition
from langchain.text_splitter import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import requests


c:\Users\zyzai\miniconda3\envs\diss_proj\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def extract_text_from_pptx(file_path):
    try:
        prs = Presentation(file_path)
        text = [shape.text for slide in prs.slides for shape in slide.shapes if hasattr(shape, "text")]
        return "\n".join(text)
    except Exception:
        return None

def extract_text_from_docx(file_path):
    try:
        doc = Document(file_path)
        return "\n".join([para.text for para in doc.paragraphs])
    except Exception:
        return None

def extract_text_with_unstructured(file_path):
    try:
        elements = partition(file_path=file_path)
        return "\n".join([str(el) for el in elements])
    except Exception:
        return None

def extract_text(file_path):
    ext = Path(file_path).suffix.lower()
    if ext == ".pptx":
        text = extract_text_from_pptx(file_path)
    elif ext == ".docx":
        text = extract_text_from_docx(file_path)
    elif ext == ".pdf":
        text = extract_text_with_unstructured(file_path)
    else:
        text = None
    return text or extract_text_with_unstructured(file_path) or ""


In [3]:
# text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=700, chunk_overlap=150
)
base_path = Path("Diss_doc")  # replace with actual path

all_chunks = []
for folder in base_path.iterdir():
    if folder.is_dir():
        for file in folder.rglob("*"):
            if file.suffix.lower() in [".pptx", ".docx", ".pdf"]:
                raw_text = extract_text(file)
                if raw_text:
                    chunks = text_splitter.split_text(raw_text)
                    for chunk in chunks:
                        all_chunks.append({
                            "content": chunk,
                            "source": f"{folder.name}/{file.name}"
                        })


In [4]:
model = SentenceTransformer("all-MiniLM-L6-v2")
texts = [c["content"] for c in all_chunks]
metas = [c["source"] for c in all_chunks]

embeddings = model.encode(texts, show_progress_bar=True, normalize_embeddings=True)

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(np.array(embeddings))

def search(query, top_k=5):
    query_vec = model.encode([query], normalize_embeddings=True)
    scores, indices = index.search(np.array(query_vec), top_k)
    for i, idx in enumerate(indices[0]):
        print(f"\n--- Match {i+1} ---")
        print(f"Cosine Similarity: {scores[0][i]:.4f}")
        print(f"Source: {metas[idx]}")
        print(f"Content:\n{texts[idx][:500]}...")

search("project discovery phase for a fintech client")


Batches: 100%|██████████| 12/12 [00:01<00:00,  7.31it/s]



--- Match 1 ---
Cosine Similarity: 0.5304
Source: Proposals/Copy of Informa Procurement Technology Cost Optimisation - Savings Discovery v2.pptx
Content:
Informa 
Procurement Technology Cost Optimisation
Savings Discovery Project
Proposal
May 2023

Our understanding of your challenge
Our proposal
Informa TS&S have been tasked with identifying £50m of savings from a run rate of ~£350m per year operating costs, on an ongoing basis for the next three years. The TCO programme has been launched to manage this end-to-end, and the Technology Sourcing (Procurement) organisation are looking to mobilise a workstream (TP2) within this to identify, size and ...

--- Match 2 ---
Cosine Similarity: 0.4894
Source: Proposals/Copy of P1+ Discovery way of working proposal.pptx
Content:
Product Discovery WoW - Context​
Problem statement​
​Our Product Discovery process in not running efficiently and is impacting our ability to work at pace in turning ideas into valuable requirements that will make a dif

In [8]:
# === History DB Setup ===
DB_DIR = Path("Diss_doc")  # same folder as main DB
DB_DIR.mkdir(parents=True, exist_ok=True)

HIST_INDEX_PATH = DB_DIR / "history.index"
HIST_TEXTS_PATH = DB_DIR / "history_texts.npy"
HIST_METAS_PATH = DB_DIR / "history_metas.npy"

hist_dim = embeddings.shape[1]  # match doc index embedding size
hist_index = faiss.IndexFlatIP(hist_dim)
hist_texts, hist_metas = [], []

if HIST_INDEX_PATH.exists():
    hist_index = faiss.read_index(str(HIST_INDEX_PATH))
    hist_texts = np.load(HIST_TEXTS_PATH, allow_pickle=True).tolist()
    hist_metas = np.load(HIST_METAS_PATH, allow_pickle=True).tolist()


In [9]:
def save_history_db():
    faiss.write_index(hist_index, str(HIST_INDEX_PATH))
    np.save(HIST_TEXTS_PATH, np.array(hist_texts, dtype=object), allow_pickle=True)
    np.save(HIST_METAS_PATH, np.array(hist_metas, dtype=object), allow_pickle=True)

def _history_summary(query, response, max_chars=1000):
    """Simple truncation summary to control token usage."""
    return f"Q: {query}\nA: {response}"[:max_chars]

def add_history_entry(query, response, meta=None):
    text = _history_summary(query, response)
    vec = model.encode([text], normalize_embeddings=True)
    hist_index.add(np.array(vec))
    hist_texts.append(text)
    hist_metas.append(meta or {})
    save_history_db()


In [5]:
def query_mistral(prompt, model="mistral:latest"):
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={"model": model, "prompt": prompt, "stream": False}
    )
    return response.json()["response"]

def build_prompt(query, context_chunks_with_sources):
    context_text = "\n<doc>\n".join(
        f"[{src}]\n{chunk}" for chunk, src in context_chunks_with_sources
    ) + "\n</doc>"
    prompt = f"""You are a senior consultant leading the discovery phase of a client project.

You are given excerpts from past project documents. These are strictly your ONLY source of information. Do NOT use any external knowledge.

Each excerpt starts with a filename in [brackets], followed by content. Stay within the facts only. Do NOT guess, assume, generalize, or invent anything that is not explicitly written.

--- START OF CONTEXT ---

{context_text}

--- END OF CONTEXT ---

Client request:
"{query}"

Your task:

- List all clear, stated client requirements from the request (not inferred ones).
- Find similar past projects based only on visible matches in the excerpts.
- For each similar project, include:
  - The [filename]
  - A short fact-based summary of what was done
- Mention tools, industries, or outcomes only if they appear directly in the context.
- Do NOT fabricate greetings, summaries, advice, or project plans.
- Output strictly in this bullet list format:
  - [filename]: short factual statement
  - identify the technologies used in similar previous projects
  - industries involved
  - a basic timeframe

Respond only with the bullet list. Nothing else.
"""
    return prompt


In [12]:
def rag(query, top_k=10, min_score=0.5, max_context_chunks=8, top_k_hist=3):
    # 1️⃣ Search document index
    query_vec = model.encode([query], normalize_embeddings=True)
    scores, indices = index.search(np.array(query_vec), top_k)

    context_chunks_with_sources = []
    seen_sources = set()
    for i, idx in enumerate(indices[0]):
        if scores[0][i] >= min_score and metas[idx] not in seen_sources:
            context_chunks_with_sources.append((texts[idx], metas[idx]))
            seen_sources.add(metas[idx])
            if len(context_chunks_with_sources) >= max_context_chunks:
                break

    # 2️⃣ Search history index
    if len(hist_texts) > 0:
        hist_scores, hist_indices = hist_index.search(np.array(query_vec), top_k_hist)
        for i, h_idx in enumerate(hist_indices[0]):
            if hist_scores[0][i] >= min_score:
                context_chunks_with_sources.insert(
                    0, (hist_texts[h_idx], f"History_{h_idx}")
                )

    if not context_chunks_with_sources:
        return "Not enough info."

    # 3️⃣ Build prompt and get response
    prompt = build_prompt(query, context_chunks_with_sources)
    response = query_mistral(prompt)

    # 4️⃣ Save new turn to history
    add_history_entry(query, response, meta={"type": "rag_turn"})

    return response


In [ ]:
# rag(
"""From:
Anita Desai
Head of Digital Services
City of Riverton Council

Subject:
Request for Support on Citizen Services Modernization Initiative

Hi team,

We’re reaching out to explore working with your consultancy on a new initiative we're planning.

The City of Riverton is looking to modernize how we deliver public services online. Our current systems are fragmented and heavily manual — for example, applying for permits, reporting local issues, and accessing social support all happen on different platforms (some even still on paper).

Our aim is to create a unified digital experience for citizens. This would include:

    A central online portal where people can log in and access multiple services

    Backend automation to reduce manual workloads for our staff

    Better tracking and analytics to understand service usage and pain points

We’re particularly keen to learn what similar projects you’ve worked on with other public sector bodies, and how we can avoid common pitfalls. We know this will be a multi-phase project, and right now we’re just trying to get a clear picture of what the discovery and planning would look like.

Looking forward to hearing how you’d approach this.

Thanks,
Anita
    """
# )


 Client Requirements:
- A central online portal where people can log in and access multiple services
- Backend automation to reduce manual workloads for staff
- Better tracking and analytics to understand service usage and pain points

Similar Past Projects:
- Proposals/Copy of FINAL ANSWERS TO BE REVIEWED FOR FORMATTING.docx: The project involved the delivery of a full technology stack implementation for Reuters Media, which included Martech (Eloqua), CRM (Salesforce) and Billing (Zuora). Although the project was not in the public sector, it does show an example of delivering end-to-end journeys and delivering at pace.
- BA&S Previous Project Review and Toolkit - Draft - 2023.pptx: Clarasys delivered a detailed current state assessment looking at all internal services, processes and handovers, customer touchpoints and insights, ways of working and technology related challenges for St John Ambulance. The aim was to create a consolidated and aligned service based on streamlined operatio

In [13]:
# === Continuous Historical Chatbot Loop ===
print("Historical RAG Chatbot. Type 'exit' to quit.\n")

while True:
    user_q = input("You: ").strip()
    if user_q.lower() in ["exit", "quit"]:
        print("Chatbot: Goodbye!")
        break

    bot_resp = rag(user_q)
    print("\nChatbot:", bot_resp, "\n")


Historical RAG Chatbot. Type 'exit' to quit.


Chatbot:  Client requirements:
1. Create a unified digital experience for citizens
2. Develop a central online portal where people can log in and access multiple services
3. Implement backend automation to reduce manual workloads for staff
4. Improve tracking and analytics to understand service usage and pain points

Similar past projects:
1. Proposals/Copy of FINAL ANSWERS TO BE REVIEWED FOR FORMATTING.docx: Project engaged in full Technology Stack Implementation for a globally distributed partner network, including Martech (Eloqua), CRM (Salesforce) and Billing (Zuora).
2. Immediate Media Digital Transformation  - Handover deck_.pptx: A project that aimed to create a unified experience for print and digital journeys, with a focus on market leading capabilities and backend automation. 


Chatbot: Not enough info. 


Chatbot: Not enough info. 

Chatbot: Goodbye!
